# Genre Classification with Transformers

This notebook fine-tunes a pretrained transformer encoder to predict a song's genre from its lyrics. It uses the transformer input created in `preprocessing_transformers.ipynb`:
- `lyrics_classification`: lightly cleaned lyrics with repeated lines removed and head + tail truncation for songs over 510 tokens,
- `data/validation_ids.parquet`: the shared stratified 25% validation split, so the results are comparable with the other models of the project.

## Setup

In [1]:
import random
from pathlib import Path

import numpy as np
import polars as pl
import torch

In [2]:
SEED = 80085
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Apple GPU locally, CUDA on Colab, CPU otherwise
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

device: cuda


In [3]:
# The project root, which holds data/ and models/. On Colab, set it to the uploaded project folder
project_dir = Path("..")

data_dir = project_dir / "data"
tran_dir = data_dir / "data-transformers"
model_dir = project_dir / "models" / "transformers" / "distilroberta-genre"

## Loading the Data

The labelled songs are split into training and validation set by song `id`. Only the columns needed for classification are kept; the test set has no `tag` and is only used for the Kaggle submission.

In [4]:
songs = pl.read_parquet(tran_dir / "transformers_train.parquet", columns=["id", "tag", "lyrics_classification"])
df_test = pl.read_parquet(tran_dir / "transformers_test.parquet", columns=["id", "lyrics_classification"])
val_ids = pl.read_parquet(data_dir / "validation_ids.parquet")["id"]

df_train = songs.filter(~pl.col("id").is_in(val_ids.implode()))
df_val = songs.filter(pl.col("id").is_in(val_ids.implode()))
print(f"train: {len(df_train):,}  validation: {len(df_val):,}  test: {len(df_test):,}")

train: 101,225  validation: 33,742  test: 33,742


The genres are encoded as integer labels in alphabetical order; `id2label` maps them back for the evaluation and the submission.

In [5]:
genres = sorted(songs["tag"].unique())
label2id = {genre: k for k, genre in enumerate(genres)}
id2label = {k: genre for genre, k in label2id.items()}

df_train = df_train.with_columns(label=pl.col("tag").replace_strict(label2id))
df_val = df_val.with_columns(label=pl.col("tag").replace_strict(label2id))
label2id

{'country': 0, 'misc': 1, 'pop': 2, 'rap': 3, 'rb': 4, 'rock': 5}

In [6]:
(df_train["tag"].value_counts(normalize=True).rename({"proportion": "train"})
    .join(df_val["tag"].value_counts(normalize=True).rename({"proportion": "validation"}), on="tag")
    .with_columns(pl.col("train", "validation").round(3))
    .sort("train", descending=True))

tag,train,validation
str,f64,f64
"""pop""",0.413,0.413
"""rap""",0.286,0.286
"""rock""",0.188,0.188
"""rb""",0.046,0.046
"""misc""",0.042,0.042
"""country""",0.026,0.026


## Tokenization

The lyrics are tokenized with the tokenizer of the pretrained model, the same one that was used in `preprocessing_transformers.ipynb` to cut the songs to 510 tokens. With the two special tokens `<s>` and `</s>` every song therefore fits into 512 tokens; `truncation=True` is only a safeguard.

The songs are not padded here. Most songs are much shorter than 512 tokens, so padding happens per batch in the `DataLoader` (dynamic padding), only up to the longest song of the batch.

In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "distilroberta-base"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(texts, truncation=True, max_length=MAX_LENGTH)["input_ids"]

train_input_ids = tokenize(df_train["lyrics_classification"].to_list())
val_input_ids = tokenize(df_val["lyrics_classification"].to_list())
test_input_ids = tokenize(df_test["lyrics_classification"].to_list())

In [8]:
lengths = pl.DataFrame({"split": ["train", "validation", "test"],
                        "input_ids": [train_input_ids, val_input_ids, test_input_ids]})
(lengths.explode("input_ids")
    .with_columns(n_tokens=pl.col("input_ids").list.len())
    .group_by("split", maintain_order=True)
    .agg(median=pl.col("n_tokens").median(),
         mean=pl.col("n_tokens").mean().round(1),
         max=pl.col("n_tokens").max(),
         share_at_512=(pl.col("n_tokens") == MAX_LENGTH).mean().round(3)))

split,median,mean,max,share_at_512
str,f64,f64,u32,f64
"""train""",241.0,278.1,512,0.143
"""validation""",241.0,278.3,512,0.143
"""test""",241.0,278.5,512,0.144


## PyTorch Dataset

`LyricsDataset` returns the token ids of one song and, for training and validation, its label. `DataCollatorWithPadding` pads each batch and adds the attention mask, so the model ignores the padding tokens.

14% of the songs are 512 tokens long, so almost every random batch of 16 contains one and is padded to 512. To pad less, `LengthGroupedSampler` puts songs of similar length into the same batch: the songs are shuffled, cut into chunks of 50 batches and sorted by length within each chunk, and the batches are then shuffled again, so the training order stays random. Validation and test songs are simply sorted by length; `predict` puts the predictions back into the original order, so they can be matched to the song `id`s.

In [9]:
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorWithPadding


class LyricsDataset(Dataset):
    def __init__(self, input_ids, labels=None):
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {"input_ids": self.input_ids[idx]}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item


train_dataset = LyricsDataset(train_input_ids, df_train["label"].to_list())
val_dataset = LyricsDataset(val_input_ids, df_val["label"].to_list())
test_dataset = LyricsDataset(test_input_ids)

In [10]:
import math
from torch.utils.data import Sampler

BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64


class LengthGroupedSampler(Sampler):
    def __init__(self, lengths, batch_size, chunk_batches=50, generator=None):
        self.lengths = lengths
        self.batch_size = batch_size
        self.chunk_size = batch_size * chunk_batches
        self.generator = generator

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)

    def __iter__(self):
        order = torch.randperm(len(self.lengths), generator=self.generator).tolist()
        batches = []
        for start in range(0, len(order), self.chunk_size):
            chunk = sorted(order[start:start + self.chunk_size], key=lambda i: self.lengths[i])
            batches += [chunk[i:i + self.batch_size] for i in range(0, len(chunk), self.batch_size)]
        for k in torch.randperm(len(batches), generator=self.generator).tolist():
            yield batches[k]


def sorted_loader(dataset, lengths):
    order = np.argsort(lengths, kind="stable")
    batches = [order[i:i + EVAL_BATCH_SIZE].tolist() for i in range(0, len(order), EVAL_BATCH_SIZE)]
    return DataLoader(dataset, batch_sampler=batches, collate_fn=collator)


collator = DataCollatorWithPadding(tokenizer)
generator = torch.Generator().manual_seed(SEED)

train_lengths = [len(ids) for ids in train_input_ids]
train_loader = DataLoader(train_dataset, collate_fn=collator,
                          batch_sampler=LengthGroupedSampler(train_lengths, BATCH_SIZE, generator=generator))
val_loader = sorted_loader(val_dataset, [len(ids) for ids in val_input_ids])
test_loader = sorted_loader(test_dataset, [len(ids) for ids in test_input_ids])
print(f"batches per epoch: {len(train_loader):,} train, {len(val_loader):,} validation, {len(test_loader):,} test")

batches per epoch: 6,327 train, 528 validation, 528 test


A quick check of one training batch: the tensor shapes, the labels, and the beginning of the first song decoded back to text.

In [11]:
batch = next(iter(train_loader))
for name, tensor in batch.items():
    print(f"{name:15} {tuple(tensor.shape)}  {tensor.dtype}")
print("labels:", [id2label[k] for k in batch["labels"].tolist()])
print()
print(tokenizer.decode(batch["input_ids"][0][:60]))

input_ids       (16, 268)  torch.int64
labels          (16,)  torch.int64
attention_mask  (16, 268)  torch.int64
labels: ['rock', 'rap', 'pop', 'rock', 'rap', 'rock', 'country', 'pop', 'rock', 'pop', 'rock', 'country', 'pop', 'rb', 'pop', 'pop']

<s>Everybody get out on the floor
We ain't gonna live this lie no more
Baby if you feel it raise your hand
You ain't alone just shout it with the band - Sanctuary

America's on trial
There's been convictions but no crime
Our government's conspired



## Model

`distilroberta-base` (Sanh et al., 2019) is a 6-layer distilled version of RoBERTa (Liu et al., 2019): about twice as fast as RoBERTa-base while keeping most of its accuracy, which makes fine-tuning on 100k songs feasible on a Colab GPU. `AutoModelForSequenceClassification` adds a randomly initialised classification head with one output per genre on top of the pretrained encoder.

In [12]:
from transformers import AutoModelForSequenceClassification


def load_model(name_or_path):
    return AutoModelForSequenceClassification.from_pretrained(
        name_or_path, num_labels=len(genres), id2label=id2label, label2id=label2id).to(device)

### Weighted Loss

The genres are very imbalanced (pop 41%, country 2.6%). With a plain cross-entropy loss the model can do well by mostly predicting the large genres. Every class is therefore weighted by the inverse of its frequency in the training set (`n_songs / (n_genres * n_songs_of_genre)`, the `"balanced"` weights of scikit-learn), so all genres contribute equally to the loss.

In [13]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight("balanced", classes=np.arange(len(genres)), y=df_train["label"].to_numpy())
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
dict(zip(genres, class_weights.round(2)))

{'country': np.float64(6.49),
 'misc': np.float64(3.99),
 'pop': np.float64(0.4),
 'rap': np.float64(0.58),
 'rb': np.float64(3.63),
 'rock': np.float64(0.89)}

## Training

The model is fine-tuned with AdamW, a learning rate of 2e-5 with a linear warm-up over the first 6% of the steps followed by a linear decay, and gradient clipping at 1.0, the fine-tuning setup of Liu et al. (2019). On a CUDA GPU the forward pass runs in mixed precision (float16), which roughly halves the training time.

**Tracking.** The average training loss is logged every `LOG_EVERY` steps. After every epoch the model is evaluated on the validation set (weighted loss, accuracy, macro-F1), and everything is saved to `history.json`.

**Saving.** The model with the best validation macro-F1 is saved to `model_dir/best`. In addition, a checkpoint with the model, optimizer and scheduler state is written after every epoch: if Colab disconnects, re-running the notebook continues after the last finished epoch.

**Switches.** `TRAIN = False` skips the training and loads the saved history and model, so the notebook can be run from start to finish without a GPU. `QUICK_RUN = True` trains for only 300 steps into a separate folder, to check that the whole pipeline works before starting the full run.

In [14]:
TRAIN = True        # False: load the saved model and training history instead of training
QUICK_RUN = False   # True: 300 training steps into a separate folder, to test the pipeline

EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
LOG_EVERY = 200
QUICK_STEPS = 300

if QUICK_RUN:
    EPOCHS = 1
    model_dir = model_dir.parent / "quick-run"

best_dir = model_dir / "best"
checkpoint_path = model_dir / "checkpoint.pt"
history_path = model_dir / "history.json"

In [15]:
import json
import time

import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup

use_amp = device.type == "cuda"


def autocast():
    return torch.autocast(device.type, dtype=torch.float16, enabled=use_amp)


@torch.no_grad()
def predict(model, loader):
    """Logits of all songs of a `sorted_loader`, in the original order of the dataset."""
    model.eval()
    logits = []
    for batch in tqdm(loader, desc="predicting", leave=False):
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with autocast():
            logits.append(model(**batch).logits.float().cpu())
    logits = torch.cat(logits)
    order = [i for batch in loader.batch_sampler for i in batch]
    restored = torch.empty_like(logits)
    restored[order] = logits
    return restored.numpy()


def evaluate(model, loader, labels):
    logits = predict(model, loader)
    preds = logits.argmax(axis=1)
    loss = F.cross_entropy(torch.from_numpy(logits), torch.tensor(labels), weight=loss_fn.weight.cpu()).item()
    return {"val_loss": loss,
            "val_accuracy": accuracy_score(labels, preds),
            "val_macro_f1": f1_score(labels, preds, average="macro")}

In [16]:
def train():
    model_dir.mkdir(parents=True, exist_ok=True)
    model = load_model(MODEL_NAME)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    history = {"steps": [], "epochs": []}
    start_epoch = 0

    # Resume after a disconnect: continue after the last finished epoch
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        generator.set_state(checkpoint["generator"].cpu())
        history = checkpoint["history"]
        start_epoch = checkpoint["epoch"] + 1
        print(f"resuming after epoch {start_epoch}")
    best_f1 = max((e["val_macro_f1"] for e in history["epochs"]), default=-1.0)

    val_labels = df_val["label"].to_list()
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        start = time.time()
        epoch_loss, logged_loss = 0.0, 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}"), start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            with autocast():
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            epoch_loss += loss.item()
            logged_loss += loss.item()
            if step % LOG_EVERY == 0:
                history["steps"].append({"epoch": epoch + 1, "step": epoch * len(train_loader) + step,
                                         "train_loss": logged_loss / LOG_EVERY,
                                         "learning_rate": scheduler.get_last_lr()[0]})
                logged_loss = 0.0
            if QUICK_RUN and step == QUICK_STEPS:
                break

        metrics = evaluate(model, val_loader, val_labels)
        history["epochs"].append({"epoch": epoch + 1, "train_loss": epoch_loss / step, **metrics,
                                  "minutes": (time.time() - start) / 60})
        print(", ".join(f"{k}: {v:.4f}" for k, v in history["epochs"][-1].items()))

        if metrics["val_macro_f1"] > best_f1:
            best_f1 = metrics["val_macro_f1"]
            model.save_pretrained(best_dir)
            tokenizer.save_pretrained(best_dir)
            print(f"saved new best model (macro-F1 {best_f1:.4f})")

        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "generator": generator.get_state(), "history": history, "epoch": epoch},
                   checkpoint_path)
        history_path.write_text(json.dumps(history, indent=1))
    return history

In [17]:
if TRAIN:
    history = train()
else:
    history = json.loads(history_path.read_text())

pl.DataFrame(history["epochs"]).with_columns(pl.exclude("epoch").round(4))

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


resuming after epoch 2


epoch 3/3:   0%|          | 0/6327 [00:00<?, ?it/s]

predicting:   0%|          | 0/528 [00:00<?, ?it/s]

epoch: 3.0000, train_loss: 0.7876, val_loss: 1.0079, val_accuracy: 0.6267, val_macro_f1: 0.5870, minutes: 16.2760


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved new best model (macro-F1 0.5870)


epoch,train_loss,val_loss,val_accuracy,val_macro_f1,minutes
i64,f64,f64,f64,f64,f64
1,1.0804,1.0252,0.6056,0.5646,16.0881
2,0.8937,0.9952,0.6046,0.5682,16.1004
3,0.7876,1.0079,0.6267,0.587,16.276


The model with the best validation macro-F1 is loaded for the evaluation.

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]